In [1]:
# ============================================================
# IMPORT LIBRARIES
# ============================================================

from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import datasets
from torchvision import transforms
from torchvision import models

from torch.utils.data import DataLoader

print("Libraries Imported Successfully")

Libraries Imported Successfully


In [2]:
# ============================================================
# DEVICE CONFIGURATION
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


# ============================================================
# DATASET PATHS
# ============================================================

PROJECT_ROOT = Path.cwd().parent

TRAIN_DIR = PROJECT_ROOT / "data" / "train"
VAL_DIR = PROJECT_ROOT / "data" / "validation"
TEST_DIR = PROJECT_ROOT / "data" / "test"

print("\nTrain Path:", TRAIN_DIR)
print("Validation Path:", VAL_DIR)
print("Test Path:", TEST_DIR)

Using Device: cuda
GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU

Train Path: d:\Railway_AI_Inspector\data\train
Validation Path: d:\Railway_AI_Inspector\data\validation
Test Path: d:\Railway_AI_Inspector\data\test


In [3]:
# ============================================================
# IMAGE TRANSFORMATIONS
# ============================================================

transform = transforms.Compose([

    transforms.Resize((224, 224)),

    transforms.RandomHorizontalFlip(),

    transforms.RandomRotation(10),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )

])

print("ResNet18 Transformations Created Successfully")

ResNet18 Transformations Created Successfully


In [4]:
# ============================================================
# CREATE DATASETS
# ============================================================

train_dataset = datasets.ImageFolder(
    root=TRAIN_DIR,
    transform=transform
)

val_dataset = datasets.ImageFolder(
    root=VAL_DIR,
    transform=transform
)

test_dataset = datasets.ImageFolder(
    root=TEST_DIR,
    transform=transform
)

print("Train Samples:", len(train_dataset))
print("Validation Samples:", len(val_dataset))
print("Test Samples:", len(test_dataset))

print("\nClasses:")
print(train_dataset.classes)

Train Samples: 70
Validation Samples: 15
Test Samples: 15

Classes:
['broken_rail', 'crack', 'misalignment', 'normal', 'surface_wear']


In [5]:
# ============================================================
# CREATE DATALOADERS
# ============================================================

BATCH_SIZE = 16

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("DataLoaders Created Successfully")
print("Batch Size:", BATCH_SIZE)

DataLoaders Created Successfully
Batch Size: 16


In [6]:
# ============================================================
# VERIFY FIRST BATCH
# ============================================================

images, labels = next(iter(train_loader))

print("Images Shape :", images.shape)
print("Labels Shape :", labels.shape)

print("\nSample Labels:")
print(labels)

Images Shape : torch.Size([16, 3, 224, 224])
Labels Shape : torch.Size([16])

Sample Labels:
tensor([4, 2, 3, 4, 3, 2, 3, 1, 1, 2, 0, 0, 1, 4, 2, 0])


In [7]:
# ============================================================
# LOAD PRETRAINED RESNET18
# ============================================================

resnet18 = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

print("Pretrained ResNet18 Loaded Successfully")

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to C:\Users\Sachin Chaudhari/.cache\torch\hub\checkpoints\resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:12<00:00, 3.73MB/s]

Pretrained ResNet18 Loaded Successfully


In [8]:
# ============================================================
# MODIFY FINAL LAYER
# ============================================================

num_features = resnet18.fc.in_features

resnet18.fc = nn.Linear(
    num_features,
    5
)

resnet18 = resnet18.to(device)

print("Final Layer Modified Successfully")
print(resnet18.fc)

Final Layer Modified Successfully
Linear(in_features=512, out_features=5, bias=True)


In [9]:
# ============================================================
# FREEZE PRETRAINED LAYERS
# ============================================================

for param in resnet18.parameters():
    param.requires_grad = False

for param in resnet18.fc.parameters():
    param.requires_grad = True

print("Transfer Learning Setup Complete")
print("Only Final Layer Trainable")

Transfer Learning Setup Complete
Only Final Layer Trainable


In [10]:
# ============================================================
# LOSS FUNCTION & OPTIMIZER
# ============================================================

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    resnet18.fc.parameters(),
    lr=0.001
)

print("Loss Function :", criterion)
print("Optimizer :", optimizer)

Loss Function : CrossEntropyLoss()
Optimizer : Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0
)


In [11]:
# ============================================================
# TRAINING CONFIGURATION
# ============================================================

NUM_EPOCHS = 10

train_losses = []
val_losses = []

train_accuracies = []
val_accuracies = []

print("Epochs :", NUM_EPOCHS)
print("Training Tracking Initialized")

Epochs : 10
Training Tracking Initialized


In [12]:
# ============================================================
# TRAINING FUNCTION
# ============================================================

def train_one_epoch(model, loader, criterion, optimizer):

    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)

        correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / len(loader)

    epoch_accuracy = 100 * correct / total

    return epoch_loss, epoch_accuracy


print("Training Function Created Successfully")

Training Function Created Successfully


In [13]:
# ============================================================
# VALIDATION FUNCTION
# ============================================================

def validate_one_epoch(model, loader, criterion):

    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)

            running_loss += loss.item()

            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)

            correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / len(loader)

    epoch_accuracy = 100 * correct / total

    return epoch_loss, epoch_accuracy


print("Validation Function Created Successfully")

Validation Function Created Successfully


In [14]:
# ============================================================
# TRAIN RESNET18
# ============================================================

for epoch in range(NUM_EPOCHS):

    train_loss, train_acc = train_one_epoch(
        resnet18,
        train_loader,
        criterion,
        optimizer
    )

    val_loss, val_acc = validate_one_epoch(
        resnet18,
        val_loader,
        criterion
    )

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    train_accuracies.append(train_acc)
    val_accuracies.append(val_acc)

    print(
        f"Epoch [{epoch+1}/{NUM_EPOCHS}] | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.2f}% | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.2f}%"
    )

print("\nTransfer Learning Training Completed")

Epoch [1/10] | Train Loss: 1.6303 | Train Acc: 27.14% | Val Loss: 1.4017 | Val Acc: 33.33%
Epoch [2/10] | Train Loss: 1.3622 | Train Acc: 44.29% | Val Loss: 1.2616 | Val Acc: 60.00%
Epoch [3/10] | Train Loss: 1.2074 | Train Acc: 60.00% | Val Loss: 1.2869 | Val Acc: 40.00%
Epoch [4/10] | Train Loss: 1.1582 | Train Acc: 64.29% | Val Loss: 1.2281 | Val Acc: 60.00%
Epoch [5/10] | Train Loss: 1.0229 | Train Acc: 71.43% | Val Loss: 1.0958 | Val Acc: 60.00%
Epoch [6/10] | Train Loss: 0.9374 | Train Acc: 72.86% | Val Loss: 0.9673 | Val Acc: 66.67%
Epoch [7/10] | Train Loss: 0.9100 | Train Acc: 75.71% | Val Loss: 1.1878 | Val Acc: 60.00%
Epoch [8/10] | Train Loss: 0.8626 | Train Acc: 85.71% | Val Loss: 1.0711 | Val Acc: 60.00%
Epoch [9/10] | Train Loss: 0.8136 | Train Acc: 81.43% | Val Loss: 0.9824 | Val Acc: 73.33%
Epoch [10/10] | Train Loss: 0.7657 | Train Acc: 80.00% | Val Loss: 1.0424 | Val Acc: 66.67%

Transfer Learning Training Completed


In [15]:
# ============================================================
# SAVE RESNET18 MODEL
# ============================================================

MODEL_PATH = PROJECT_ROOT / "models" / "railway_resnet18_transfer.pth"

torch.save(resnet18.state_dict(), MODEL_PATH)

print("Transfer Learning Model Saved Successfully")
print("Location :", MODEL_PATH)

Transfer Learning Model Saved Successfully
Location : d:\Railway_AI_Inspector\models\railway_resnet18_transfer.pth
